# VisionAI Inspector — Stage 1: Data Analysis & Preprocessing

**Goal of this notebook:** understand the dataset and prepare it for machine
learning. We are **not** building a model yet — that's Stage 2.

Dataset: [MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad),
category `bottle` (see `README.md` for why this dataset was chosen and how to
download it).

If the real dataset isn't downloaded yet, this notebook auto-generates a tiny
synthetic placeholder dataset with the same folder structure, purely to prove
every cell runs correctly end-to-end. Any output computed from placeholder
data is clearly labeled **[SYNTHETIC PLACEHOLDER DATA]** — those numbers are
not real and should never be quoted as project results. Drop the real
`bottle/` folder into `data/raw/mvtec_ad/` and re-run to get genuine results.


## 1. Setup and imports

Standard Python project setup: imports, paths, config constants. Keeping config values (category name, target image size, split ratios) as named variables at the top — not scattered magic numbers — makes the notebook easy to re-run for a different category later.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Make src/ importable from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_utils import (
    build_metadata_dataframe,
    discover_categories,
    check_image_validity,
    load_image_as_array,
    normalize_image,
    make_synthetic_placeholder_dataset,
)

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "mvtec_ad"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CATEGORY = "bottle"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data root: {RAW_ROOT}")

## 2. Locate the dataset (real or placeholder)

We check whether the real MVTec `bottle` category is present. If it isn't, we generate the synthetic placeholder described above so the rest of the notebook can still be verified to run.

In [ ]:
category_path = RAW_ROOT / CATEGORY
USING_SYNTHETIC_DATA = not category_path.exists()

if USING_SYNTHETIC_DATA:
    print("Real MVTec data not found at:", category_path)
    print("Generating a small synthetic placeholder dataset for pipeline testing...")
    make_synthetic_placeholder_dataset(RAW_ROOT, category=CATEGORY, seed=RANDOM_SEED)
    print("Done. Replace data/raw/mvtec_ad/bottle/ with the real download, then re-run this notebook.")
else:
    print("Found real dataset at:", category_path)

DATA_TAG = "[SYNTHETIC PLACEHOLDER DATA] " if USING_SYNTHETIC_DATA else ""
print("\nAvailable categories under data/raw/mvtec_ad/:", discover_categories(RAW_ROOT))

## 2b. Official MVTec AD `bottle` structure — what it should contain, and why

Before doing any EDA on our own copy, it's worth writing down what the
**published** dataset structure actually is, independent of what's on disk
right now, so we have something to check our download against rather than
just trusting whatever we happen to have.

**Confirmed from the original MVTec AD paper (Bergmann et al., 2019) and
independent secondary sources — not fabricated, not computed from our copy:**

| Split | Folder | Count |
|---|---|---|
| `train` | `good` | **209** |
| `test` | `good` | **20** |
| `test` | `broken_large` + `broken_small` + `contamination` (defective, combined) | **63** |
| `ground_truth` | masks (one per defective test image) | **63** |
| **Total** | | **292** |

The exact split of the 63 defective test images across the three individual
defect subfolders isn't reliably documented in public secondary sources, so
we don't hard-code a number for that — section 4 below reads it directly off
disk, which is the right source of truth for that detail anyway.

**Why the training set is (almost) entirely defect-free:** MVTec AD is built
for *unsupervised / semi-supervised anomaly detection*, not standard
supervised classification. This mirrors a real constraint in industrial QC:
defect-free units are abundant and cheap to collect, but defects are rare,
highly varied, and expensive to exhaustively catalog — you can never collect
a labeled sample that covers "every way this part can break." So instead of
teaching a model what defects look like, the standard approach teaches it
what *normal* looks like and flags deviations at test time. The benchmark's
`train` = normal-only / `test` = normal + every defect type protocol exists
specifically to evaluate that paradigm fairly and comparably across papers.

This has a direct, concrete consequence for how we're allowed to split the
data — see section 11.

## 3. Build a metadata table (file handling, `pathlib`, functions)

`build_metadata_dataframe` (in `src/data_utils.py`) walks the folder tree with `pathlib` and returns one row per image: its path, MVTec's original train/test split, a binary `good`/`defective` label, and the specific `defect_type`. This is the metadata table every later stage builds on.

In [ ]:
df = build_metadata_dataframe(category_path)
print(f"{DATA_TAG}Total images found: {len(df)}")
df.head(10)

### 3b. Sanity check against the official reference counts (section 2b)

Confirms this is a genuine, complete copy of MVTec AD `bottle` before we trust any statistic computed from it. This check is meaningless on synthetic placeholder data by design — it's only checking real downloads.

In [ ]:
EXPECTED_COUNTS = {
    ("train", "good"): 209,
    ("test", "good"): 20,
}
EXPECTED_TEST_DEFECTIVE_TOTAL = 63
EXPECTED_TOTAL_IMAGES = 292

if USING_SYNTHETIC_DATA:
    print("Skipping reference-count check: this is SYNTHETIC PLACEHOLDER DATA.")
    print("It deliberately does NOT match the official counts (it's a tiny stand-in).")
    print("This check only means something once the real MVTec bottle/ folder is in place.")
else:
    print("Checking on-disk counts against the official published bottle category structure...\n")
    ok = True
    for (split, defect_type), expected in EXPECTED_COUNTS.items():
        actual = len(df[(df["split"] == split) & (df["defect_type"] == defect_type)])
        status = "OK" if actual == expected else "MISMATCH"
        if actual != expected:
            ok = False
        print(f"  {split}/{defect_type}: expected {expected}, found {actual}  [{status}]")

    actual_test_defective = len(df[(df["split"] == "test") & (df["defect_type"] != "good")])
    status = "OK" if actual_test_defective == EXPECTED_TEST_DEFECTIVE_TOTAL else "MISMATCH"
    if actual_test_defective != EXPECTED_TEST_DEFECTIVE_TOTAL:
        ok = False
    print(f"  test/<defective, combined>: expected {EXPECTED_TEST_DEFECTIVE_TOTAL}, found {actual_test_defective}  [{status}]")

    actual_total = len(df)
    status = "OK" if actual_total == EXPECTED_TOTAL_IMAGES else "MISMATCH"
    if actual_total != EXPECTED_TOTAL_IMAGES:
        ok = False
    print(f"  total images: expected {EXPECTED_TOTAL_IMAGES}, found {actual_total}  [{status}]")

    print("\n" + ("This is a complete, correctly-structured copy of MVTec AD bottle."
                    if ok else
                    "Counts don't fully match — check the download/extraction (see README) before trusting downstream stats."))

## 4. Pandas: dataset structure and class distribution

Basic DataFrame operations — `groupby`, `value_counts`, `describe` — to understand what's actually in the dataset before doing anything else.

In [ ]:
print(f"{DATA_TAG}Rows per MVTec split:")
print(df["split"].value_counts())

print(f"\n{DATA_TAG}Rows per label (good vs defective):")
print(df["label"].value_counts())

print(f"\n{DATA_TAG}Rows per defect_type:")
print(df["defect_type"].value_counts())

In [ ]:
# Cross-tab: how labels/defect types break down across the two MVTec splits.
# This matters because MVTec's train/ split is 100% "good" by design (it's
# built for unsupervised anomaly detection) — we need to see that clearly
# before deciding how to build our OWN train/val/test split in section 8.
pd.crosstab(df["split"], df["defect_type"])

## 5. Statistics: class imbalance

Mean, proportions, and an explicit imbalance ratio. Class imbalance matters because a classifier trained naively on an imbalanced dataset can get high accuracy just by always predicting the majority class — we need to know this up front so Stage 2/3 can address it (e.g. stratified splits, class weighting, or resampling).

In [ ]:
label_counts = df["label"].value_counts()
good_n = int(label_counts.get("good", 0))
defective_n = int(label_counts.get("defective", 0))
total_n = good_n + defective_n

print(f"{DATA_TAG}good: {good_n} ({good_n/total_n:.1%})")
print(f"{DATA_TAG}defective: {defective_n} ({defective_n/total_n:.1%})")
if defective_n > 0:
    print(f"{DATA_TAG}Imbalance ratio (good : defective) = {good_n/defective_n:.2f} : 1")

defect_type_counts = df.loc[df["label"] == "defective", "defect_type"].value_counts()
print(f"\n{DATA_TAG}Defect type distribution (within 'defective' only):")
print(defect_type_counts)

## 6. Visualization: class and defect-type distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_counts.plot(kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"])
axes[0].set_title(f"{DATA_TAG}good vs defective")
axes[0].set_xlabel("label")
axes[0].set_ylabel("count")
axes[0].tick_params(axis="x", rotation=0)

defect_type_counts.plot(kind="bar", ax=axes[1], color="#C44E52")
axes[1].set_title(f"{DATA_TAG}defect type distribution")
axes[1].set_xlabel("defect_type")
axes[1].set_ylabel("count")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 7. Data quality: corrupt / invalid image check

Before trusting any of these images, verify each one actually opens and isn't truncated or corrupt. `check_image_validity` tries to open and decode every file and records the result instead of crashing — this is the kind of defensive check real pipelines need for data that came from scraping, shipping, or repeated copying.

In [ ]:
validity_rows = df["filepath"].apply(check_image_validity).apply(pd.Series)
df = pd.concat([df, validity_rows], axis=1)

n_invalid = (~df["is_valid"]).sum()
print(f"{DATA_TAG}Invalid/corrupt images: {n_invalid} out of {len(df)}")
if n_invalid > 0:
    print(df.loc[~df["is_valid"], ["filepath", "error"]])

# Drop invalid rows so later stages never touch them
df = df[df["is_valid"]].reset_index(drop=True)

print(f"\n{DATA_TAG}Image dimension summary (width / height in pixels):")
df[["width", "height"]].describe()

## 8. Visualization: sample images per class

Always look at actual samples, not just statistics — this is how you'd catch mislabeled folders, unexpected color modes, or a defect type that looks nothing like its name suggests.

In [ ]:
classes_to_show = df["defect_type"].unique().tolist()
n_cols = len(classes_to_show)
n_samples_per_class = 3

fig, axes = plt.subplots(n_samples_per_class, n_cols, figsize=(3 * n_cols, 3 * n_samples_per_class))
fig.suptitle(f"{DATA_TAG}Sample images per defect_type", y=1.02)

for col, defect_type in enumerate(classes_to_show):
    subset = df[df["defect_type"] == defect_type]["filepath"].tolist()
    for row in range(n_samples_per_class):
        ax = axes[row, col] if n_cols > 1 else axes[row]
        if row < len(subset):
            img = load_image_as_array(subset[row])
            ax.imshow(img)
        ax.axis("off")
        if row == 0:
            ax.set_title(defect_type, fontsize=10)

plt.tight_layout()
plt.show()

## 9. NumPy: image representation, shape, and pixel statistics

An image loaded with PIL and converted to a NumPy array has shape `(height, width, channels)`. We inspect dtype, value range, and pixel intensity distribution — this is the foundation for the normalization step that follows.

In [ ]:
sample_path = df.iloc[0]["filepath"]
sample_img = load_image_as_array(sample_path)

print(f"{DATA_TAG}Array shape (H, W, C): {sample_img.shape}")
print(f"{DATA_TAG}dtype: {sample_img.dtype}")
print(f"{DATA_TAG}Pixel value range: [{sample_img.min()}, {sample_img.max()}]")
print(f"{DATA_TAG}Per-channel mean (R, G, B): {sample_img.reshape(-1, 3).mean(axis=0)}")
print(f"{DATA_TAG}Per-channel std  (R, G, B): {sample_img.reshape(-1, 3).std(axis=0)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(sample_img)
axes[0].set_title(f"{DATA_TAG}Sample image")
axes[0].axis("off")

colors = ["red", "green", "blue"]
for i, c in enumerate(colors):
    axes[1].hist(sample_img[:, :, i].ravel(), bins=50, alpha=0.5, color=c, label=c)
axes[1].set_title(f"{DATA_TAG}Pixel intensity histogram (0-255)")
axes[1].set_xlabel("pixel value")
axes[1].set_ylabel("frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Preprocessing: resizing and normalization

Two separate, explicit steps:

1. **Resize** every image to a fixed size — models need a consistent input shape, and raw industrial photos rarely come pre-sized.
2. **Normalize** pixel values — see the explanation inside `normalize_image` in `src/data_utils.py`. We show both the `[0, 1]` scaling and ImageNet mean/std normalization (the latter matters once Stage 3 uses a pretrained CNN backbone).

In [ ]:
TARGET_SIZE = (224, 224)  # standard input size for many pretrained CNNs (Stage 3)

resized = load_image_as_array(sample_path, resize_to=TARGET_SIZE)
norm_01 = normalize_image(resized, method="0_1")
norm_meanstd = normalize_image(resized, method="mean_std")

print(f"{DATA_TAG}Resized shape: {resized.shape}, dtype: {resized.dtype}")
print(f"{DATA_TAG}[0,1] normalized -> range [{norm_01.min():.3f}, {norm_01.max():.3f}], dtype: {norm_01.dtype}")
print(f"{DATA_TAG}mean/std normalized -> range [{norm_meanstd.min():.3f}, {norm_meanstd.max():.3f}], dtype: {norm_meanstd.dtype}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(resized); axes[0].set_title(f"{DATA_TAG}resized (uint8)"); axes[0].axis("off")
axes[1].imshow(norm_01); axes[1].set_title(f"{DATA_TAG}[0,1] normalized"); axes[1].axis("off")
# mean/std-normalized arrays have negative values and aren't directly displayable as an image;
# we clip+rescale purely for visualization, the underlying array passed to a model stays as-is.
display_meanstd = np.clip((norm_meanstd - norm_meanstd.min()) / (norm_meanstd.max() - norm_meanstd.min()), 0, 1)
axes[2].imshow(display_meanstd); axes[2].set_title(f"{DATA_TAG}mean/std normalized\n(rescaled for display only)"); axes[2].axis("off")
plt.tight_layout()
plt.show()

## 11. Train / validation / test split — preserving the MVTec AD protocol

**Correction from an earlier version of this notebook:** the first draft of
this section pooled `train/good` and every `test/*` folder together and
re-split them randomly (stratified by defect type) into a fresh 70/15/15
train/val/test. That was methodologically wrong for this dataset, for two
concrete reasons:

1. It destroys the property that defines this as an anomaly-detection
   benchmark — that `train` contains only normal images. Mixing defect
   images into training changes what kind of problem we're even solving.
2. It leaks official **test** images into our **train** split. Every
   published MVTec AD result treats the full `test/` folder as held-out
   evaluation data. If we train on part of it, our "test performance" is no
   longer comparable to any benchmark, and is inflated relative to a fair
   evaluation — the model would have seen same-distribution defect examples
   during training that it isn't supposed to see until evaluation.

**The fix — preserve MVTec's original structure, and only add what's
missing (a validation set) without touching test:**

- **`test` stays exactly as MVTec ships it** — every `good` and every
  defective image, completely untouched, never used for training or for
  tuning any decision. This is what makes later evaluation numbers
  meaningful and comparable to published results.
- **`train` and `val` are carved only out of MVTec's original `train/good`
  pool** (209 normal images in the real dataset). We hold out a fraction of
  *just the normal training images* as validation — useful for tuning
  hyperparameters or an anomaly-score threshold — without ever exposing a
  single test image, defective or not, before final evaluation.
- The MVTec `split` column from section 3 is preserved unchanged in the
  saved output, so this decision is always traceable later.

**Open question left for Stage 2, deliberately not resolved here:** MVTec's
official `train` folder contains no defective images at all, which is fine
for an anomaly-detection model but means a *supervised* defect-type
classifier (which Stage 2 asks for) has no labeled defective training data
under the standard protocol. Common, defensible options include (a) framing
Stage 2 as anomaly detection / one-class learning on `train/good`, evaluated
on the untouched `test` set as MVTec intends, or (b) explicitly carving a
small, clearly-labeled supervised training subset out of the test-side
defective images, documenting exactly how much of `test` that consumes and
why. That's a modeling decision for Stage 2 to make and document explicitly
— Stage 1's job is only to not make that decision *for* it by silently
mixing everything together.


In [ ]:
from sklearn.model_selection import train_test_split

VAL_FRAC_OF_TRAIN = 0.15  # fraction of MVTec's train/good pool held out as validation

# Original MVTec test set: untouched, exactly as shipped. This is the ONLY
# data we ever use for final evaluation, and it is never touched below.
test_df = df[df["split"] == "test"].reset_index(drop=True)

# MVTec's original train pool: normal images only, by construction.
mvtec_train_pool = df[df["split"] == "train"].reset_index(drop=True)
assert (mvtec_train_pool["defect_type"] == "good").all(), (
    "Expected MVTec train/ to contain only 'good' images — found otherwise, "
    "check the dataset extraction."
)

# Carve train/val ONLY from this normal-only pool. No stratify needed: every
# row here has the same defect_type ("good") by construction.
train_df, val_df = train_test_split(
    mvtec_train_pool, test_size=VAL_FRAC_OF_TRAIN, random_state=RANDOM_SEED
)

for name, split_df in [("train (normal only)", train_df), ("val (normal only)", val_df), ("test (untouched, mixed)", test_df)]:
    print(f"{DATA_TAG}{name}: {len(split_df)} images")
    print(split_df["defect_type"].value_counts().to_dict())
    print()

# Leakage checks:
# 1. No file appears in more than one split.
train_paths, val_paths, test_paths = set(train_df["filepath"]), set(val_df["filepath"]), set(test_df["filepath"])
assert not (train_paths & val_paths), "Leakage between train and val!"
assert not (train_paths & test_paths), "Leakage between train and test!"
assert not (val_paths & test_paths), "Leakage between val and test!"

# 2. The specific leakage this section exists to prevent: no image that MVTec
# ships under test/ has ended up in train or val.
original_test_paths = set(df[df["split"] == "test"]["filepath"])
assert not (train_paths & original_test_paths), "An official MVTec test image leaked into train!"
assert not (val_paths & original_test_paths), "An official MVTec test image leaked into val!"

print(f"{DATA_TAG}No overlap between splits, and no official test image appears in train/val. Protocol preserved.")

## 12. Save Stage 1 outputs

Persist the split metadata as CSVs under `data/processed/` — Stage 2 loads these directly instead of re-deriving the split, which guarantees every later stage trains/evaluates on the exact same, protocol-correct split, and specifically that it never touches `bottle_test.csv` for anything other than final evaluation.

In [ ]:
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    out_path = PROCESSED_ROOT / f"{CATEGORY}_{name}.csv"
    split_df.to_csv(out_path, index=False)
    print(f"Saved {len(split_df)} rows -> {out_path}")

print(f"\n{'SYNTHETIC PLACEHOLDER outputs saved.' if USING_SYNTHETIC_DATA else 'Real dataset outputs saved.'}")

## 13. Summary — Stage 1 complete

What we have now:

- A documented, justified dataset choice (MVTec AD, `bottle` category),
  with reference counts confirmed against published sources (section 2b)
  and checked against our own downloaded copy (section 3b).
- A reusable, tested loading/preprocessing module (`src/data_utils.py`).
- A metadata table with per-image label, defect type, dimensions, and
  validity status.
- Class distribution and imbalance statistics.
- Sample image and pixel-distribution visualizations.
- A resize + normalization pipeline (two normalization strategies).
- A **protocol-preserving** split: train/val carved only from MVTec's
  original normal-only training pool, and the original `test` set left
  completely untouched for final evaluation — persisted to
  `data/processed/`.

**Next (Stage 2):** load `data/processed/{category}_{split}.csv`. Before
building any model, Stage 2 needs to explicitly decide and document how it
will handle the fact that `train` contains no defective examples (see the
"open question" in section 11) — that decision shapes everything that
follows, so it shouldn't be made implicitly.

Reminder: if the outputs above are tagged **[SYNTHETIC PLACEHOLDER DATA]**,
download the real MVTec `bottle` category (see `README.md`), place it at
`data/raw/mvtec_ad/bottle/`, confirm section 3b reports all `OK`, and
re-run this notebook before treating any number here as a real result.
